# Phase 4 + 5 — Train the IDM on a GPU, then test it honestly

**Goal:** train on all ~70k training clips, pick the best version using **val**, then score it **once** on **test**. The test score is the number that goes in your article.

### How training works here

- **Epoch:** one full pass over the training clips (~2,200 batches of 32). We do 15.
- After every epoch we score the model on **val** (clips from sessions it never trains on) and keep the best version so far.
- We judge "best" by **macro-F1**, not accuracy. Phase 1 showed why: guessing only the 4 directions already gets ~90% accuracy. Macro-F1 averages the score of all 9 actions equally, so rare diagonals count as much as common moves.
- **Test** is touched exactly once, at the very end.

### Before you start (one time)

1. In Google Drive, make a folder **`pacman-idm`** with this inside:
```
pacman-idm/
├── src/                 <- upload __init__.py, labels.py, data.py, model.py
└── data/processed/      <- upload frames.npy, splits.npz, meta.json  (~2.7 GB)
```
2. Open this notebook in Colab: **File → Upload notebook**.
3. **Runtime → Change runtime type → T4 GPU → Save.**

### Colab survival rules

- Free sessions disconnect if the tab is closed or idle too long, and the machine gets wiped. Keep the tab open.
- The notebook saves a checkpoint to Drive after **every epoch**. If you get disconnected, reconnect and **re-run all cells**: training resumes where it stopped.

## 1. Connect to Drive and find your files

In [ ]:
import os, sys, json, time, math, shutil

try:
    import google.colab                       # only exists inside Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/pacman-idm"
else:                                           # running on your laptop instead
    PROJECT = os.path.abspath(os.getcwd())
    while not os.path.exists(os.path.join(PROJECT, "README.md")) and os.path.dirname(PROJECT) != PROJECT:
        PROJECT = os.path.dirname(PROJECT)

sys.path.insert(0, PROJECT)
DRIVE_DATA = os.path.join(PROJECT, "data", "processed")
for f in ["src/labels.py", "src/data.py", "src/model.py", "data/processed/frames.npy",
          "data/processed/splits.npz", "data/processed/meta.json"]:
    print(("OK      " if os.path.exists(os.path.join(PROJECT, f)) else "MISSING ") + f)

Everything should say **OK**. If `src/__init__.py` is missing, create an empty file with that name in `src/`.

## 2. Copy the data to the Colab machine's own disk

Reading 16 random frames per clip straight from Drive is slow (Drive is a network folder). Copying `frames.npy` to the local disk once takes a minute or two and makes every epoch much faster.

In [ ]:
if IN_COLAB:
    DATA = "/content/processed"
    os.makedirs(DATA, exist_ok=True)
    for f in ["frames.npy", "splits.npz", "meta.json"]:
        dst = os.path.join(DATA, f)
        if not os.path.exists(dst):
            t = time.time(); shutil.copy(os.path.join(DRIVE_DATA, f), dst)
            print(f"copied {f} in {time.time() - t:.0f}s")
else:
    DATA = DRIVE_DATA
print("data folder:", DATA)

## 3. Settings

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

from src.data import ClipDataset
from src.model import IDM9Way, count_params

# ---- the knobs ----
QUICK = True          # True = tiny 1-epoch test run to check everything works. Then set False for the real run.
EPOCHS = 15
BATCH = 32
LR = 1e-4             # learning rate: how big each nudge to the weights is
WEIGHT_DECAY = 1e-5   # gently pulls weights toward 0, which helps generalization
SEED = 42
RUN_NAME = "baseline_9way" + ("_quick" if QUICK else "")
# -------------------

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = dev.type == "cuda"     # mixed precision: ~2x faster on a GPU, see below
print("device:", dev, "|", torch.cuda.get_device_name(0) if dev.type == "cuda" else "no GPU (fine for QUICK, way too slow for the real run)")

CKPT_DIR = os.path.join(PROJECT, "checkpoints", RUN_NAME)   # on Drive, so it survives disconnects
os.makedirs(CKPT_DIR, exist_ok=True)
torch.manual_seed(SEED); np.random.seed(SEED)

**Mixed precision (`USE_AMP`)** does most of the math with 16-bit numbers instead of 32-bit. GPUs do that about twice as fast, and the results are practically the same. The `GradScaler` below is the safety net: it keeps tiny gradient values from rounding to zero in 16-bit.

**`QUICK = True` first.** It trains 1 epoch on 2,000 clips (a few minutes) to prove the whole pipeline runs, from loading through checkpointing to test scoring. When that works, set `QUICK = False` and **Runtime → Run all**.

## 4. Data loaders

In [ ]:
train_ds = ClipDataset(DATA, "train")
val_ds = ClipDataset(DATA, "val")
test_ds = ClipDataset(DATA, "test")
NAMES = train_ds.class_names

if QUICK:
    EPOCHS = 1
    g = np.random.default_rng(0)
    train_ds = Subset(train_ds, g.choice(len(train_ds), min(2000, len(train_ds)), replace=False))
    val_ds = Subset(val_ds, g.choice(len(val_ds), min(1000, len(val_ds)), replace=False))

workers = 2 if IN_COLAB else 0
train_loader = DataLoader(train_ds, BATCH, shuffle=True, num_workers=workers, pin_memory=USE_AMP, drop_last=True)
val_loader = DataLoader(val_ds, 128, shuffle=False, num_workers=workers, pin_memory=USE_AMP)
test_loader = DataLoader(test_ds, 128, shuffle=False, num_workers=workers, pin_memory=USE_AMP)
print(f"train {len(train_ds):,} | val {len(val_ds):,} | test {len(test_ds):,} clips | {len(train_loader):,} batches per epoch")

## 5. Model, optimizer, schedule, and resuming

- **AdamW** is the optimizer: the rule for how to nudge each weight. It adapts the step size per weight.
- **Cosine schedule:** the learning rate starts at `LR` and glides down toward 0 over the 15 epochs, like a cosine curve. Big steps early to make progress, small steps late to settle in.
- If `last.pth` exists on Drive (you got disconnected), everything is restored and training continues from the next epoch.

In [ ]:
model = IDM9Way().to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
print(f"{count_params(model):,} parameters")

start_epoch, history, best_f1 = 0, [], -1.0
last_path, best_path = os.path.join(CKPT_DIR, "last.pth"), os.path.join(CKPT_DIR, "best.pth")
if os.path.exists(last_path):
    ck = torch.load(last_path, map_location=dev, weights_only=False)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
    sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
    start_epoch, history, best_f1 = ck["epoch"], ck["history"], ck["best_f1"]
    print(f"Resuming after epoch {start_epoch} (best val macro-F1 so far {best_f1:.3f})")

## 6. Train

In [ ]:
def predict(model, loader):
    """Run the model over a loader. Returns average loss, predictions, labels."""
    model.eval()
    preds, labels, loss_sum, n = [], [], 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
            with torch.autocast(dev.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(x)
            loss_sum += F.cross_entropy(logits.float(), y, reduction="sum").item()
            n += len(y)
            preds.append(logits.argmax(1).cpu()); labels.append(y.cpu())
    return loss_sum / n, torch.cat(preds).numpy(), torch.cat(labels).numpy()


def scores(p, y):
    return {"acc": float((p == y).mean()),
            "macro_f1": float(f1_score(y, p, average="macro", labels=range(9), zero_division=0)),
            "bal_acc": float(balanced_accuracy_score(y, p))}


for epoch in range(start_epoch, EPOCHS):
    model.train()
    t0, run_loss = time.time(), 0.0
    for i, (x, y) in enumerate(train_loader, 1):
        x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(dev.type, dtype=torch.float16, enabled=USE_AMP):
            loss = F.cross_entropy(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        run_loss += loss.item()
        if i % 200 == 0:
            print(f"  epoch {epoch + 1} batch {i}/{len(train_loader)}  loss {run_loss / i:.3f}  ({time.time() - t0:.0f}s)")
    sched.step()

    val_loss, vp, vy = predict(model, val_loader)
    s = scores(vp, vy)
    row = {"epoch": epoch + 1, "train_loss": run_loss / len(train_loader), "val_loss": val_loss, **s,
           "lr": sched.get_last_lr()[0], "minutes": (time.time() - t0) / 60}
    history.append(row)
    improved = s["macro_f1"] > best_f1
    if improved:
        best_f1 = s["macro_f1"]
        torch.save(model.state_dict(), best_path)
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                "scaler": scaler.state_dict(), "epoch": epoch + 1, "history": history, "best_f1": best_f1}, last_path)
    print(f"epoch {epoch + 1:2d}/{EPOCHS} | train loss {row['train_loss']:.3f} | val loss {val_loss:.3f} | "
          f"val acc {s['acc']:.3f}  macro-F1 {s['macro_f1']:.3f}  bal-acc {s['bal_acc']:.3f} | "
          f"{row['minutes']:.1f} min" + ("  <- best, saved" if improved else ""))

**Reading the epoch lines:**
- **train loss** should fall steadily.
- **val loss** falls at first. If it starts *rising* while train loss keeps falling, the model is **overfitting**: memorizing training sessions instead of learning general rules.
- **val macro-F1** is what we steer by. Expect accuracy to look great early, while macro-F1 climbs more slowly as the model learns the rare diagonals.

## 7. Learning curves

In [ ]:
ep = [h["epoch"] for h in history]
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(ep, [h["train_loss"] for h in history], "o-", label="train")
ax[0].plot(ep, [h["val_loss"] for h in history], "o-", label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
for k in ["acc", "macro_f1", "bal_acc"]:
    ax[1].plot(ep, [h[k] for h in history], "o-", label=k)
ax[1].set_title("Validation scores"); ax[1].set_xlabel("epoch"); ax[1].set_ylim(0, 1); ax[1].legend()
plt.tight_layout(); plt.savefig(os.path.join(CKPT_DIR, "learning_curves.png"), dpi=150); plt.show()

## 8. The test set: score the best model once

We load the version that did best on **val**, and run it over **test** (3 sessions it has never seen, never used to pick anything). This is your honest number.

For context we also score a **lazy baseline** that always guesses the most common action in training. Any real model has to beat it by a lot.

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=dev))
test_loss, tp, ty = predict(model, test_loader)
test = scores(tp, ty)

base_ds = train_ds.dataset if isinstance(train_ds, Subset) else train_ds
majority = int(np.argmax(base_ds.class_counts()))
lazy = scores(np.full_like(ty, majority), ty)

print(f"{'':26s} {'accuracy':>9s} {'macro-F1':>9s} {'bal-acc':>9s}")
print(f"{'always ' + NAMES[majority]:26s} {lazy['acc']:9.3f} {lazy['macro_f1']:9.3f} {lazy['bal_acc']:9.3f}")
print(f"{'your IDM (test)':26s} {test['acc']:9.3f} {test['macro_f1']:9.3f} {test['bal_acc']:9.3f}")
print()
print(classification_report(ty, tp, labels=range(9), target_names=NAMES, digits=3, zero_division=0))

**Reading the report:** for each action,
- **precision:** when the model says LEFT, how often is it really LEFT?
- **recall:** of all real LEFT clips, how many did it catch?
- **f1:** a single blend of the two.
- **support:** how many test clips had that action.

Expect high numbers for the 4 directions and much lower ones for the diagonals.

## 9. Confusion matrix: *what* does it mix up?

Row = the true action, column = the model's guess. Each row is shown as percentages, so the diagonal is recall per action. Off-diagonal cells show the mistakes.

In [ ]:
cm = confusion_matrix(ty, tp, labels=range(9))
cm_pct = cm / cm.sum(1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(cm_pct, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(9)); ax.set_xticklabels(NAMES, rotation=45, ha="right")
ax.set_yticks(range(9)); ax.set_yticklabels(NAMES)
for i in range(9):
    for j in range(9):
        ax.text(j, i, f"{100 * cm_pct[i, j]:.0f}", ha="center", va="center", fontsize=8,
                color="white" if cm_pct[i, j] > 0.5 else "black")
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Test confusion matrix (% of each true action)")
plt.colorbar(im, fraction=0.046); plt.tight_layout()
plt.savefig(os.path.join(CKPT_DIR, "confusion_test.png"), dpi=150); plt.show()

pairs = sorted(((cm_pct[i, j], NAMES[i], NAMES[j], cm[i, j]) for i in range(9) for j in range(9) if i != j), reverse=True)
print("Biggest mix-ups (true -> predicted):")
for pct, a, b, n in pairs[:6]:
    print(f"  {a:10s} -> {b:10s} {100 * pct:5.1f}%  ({n} clips)")

**Look for a pattern in the mix-ups.** My guess: diagonals get confused with their neighbours (UPLEFT → UP or LEFT). If so, the model mostly gets *one half* of a diagonal right, which is exactly the motivation for the two-head model in Phase 6.

## 10. Save the results

In [ ]:
results = {"run": RUN_NAME, "quick": QUICK, "epochs": EPOCHS, "batch": BATCH, "lr": LR, "seed": SEED,
           "params": count_params(model), "best_val_macro_f1": best_f1,
           "test": test, "lazy_baseline": lazy, "history": history,
           "per_class_recall": {NAMES[i]: float(cm_pct[i, i]) for i in range(9)}}
json.dump(results, open(os.path.join(CKPT_DIR, "results.json"), "w"), indent=2)
print("saved to", CKPT_DIR)

## Done

**If this was the QUICK run:** the scores will be bad (1 epoch, 2,000 clips). That's expected. You're only checking that every cell ran. Now set `QUICK = False` in step 3 and do **Runtime → Run all**. The real run takes roughly 1.5–3 hours on a free T4.

**After the real run, send me:**
1. the epoch lines from step 6,
2. the test table + report from step 8,
3. the confusion matrix and the "biggest mix-ups" list from step 9.

**For `notes.md`:** minutes per epoch, the best val epoch, test macro-F1 vs the lazy baseline, and the biggest mix-ups.

Everything is also saved in `pacman-idm/checkpoints/baseline_9way/` on your Drive.